In [2]:
import gc
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split

# ==========================================
# CONFIGURATIE
# ==========================================
DATA_PATH = "transformed_data_sampled.csv"
TARGET_COL = "status"  # VERANDER DIT naar de naam van jouw doelvariabele
IS_CLASSIFICATION = True  # Zet op False als het een regressieprobleem is

# ==========================================
# 1. GEHEUGENEFFICIËNT DATA INLADEN (DOWNCASTING)
# ==========================================
print("1. Dtypes analyseren voor geheugenbesparing...")
# Lees eerst alleen de eerste 100 rijen om de datatypes te bepalen
sample = pd.read_csv(DATA_PATH, nrows=100)

dtypes = {}
for col in sample.columns:
    if sample[col].dtype == "float64":
        dtypes[col] = "float32"  # Halveert het geheugen voor decimalen
    elif sample[col].dtype == "int64":
        dtypes[col] = "int32"  # Halveert het geheugen voor hele getallen
    else:
        dtypes[col] = sample[col].dtype

print("-> Data inladen met geoptimaliseerde dtypes...")
df = pd.read_csv(DATA_PATH, dtype=dtypes)

# Splits direct in X en y
X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]

# Verwijder de originele dataframe direct uit het RAM
del df
gc.collect()

# ==========================================
# 2. TRAIN / VALIDATION SPLIT
# ==========================================
print("2. Dataset opsplitsen...")
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Verwijder de basis X en y om RAM vrij te maken
del X, y
gc.collect()

# ==========================================
# 3. OMZETTEN NAAR NATIVE XGBOOST DMATRIX
# ==========================================
print("3. DMatrix aanmaken (interne XGBoost structuur)...")
# DMatrix is vele malen lichter in het RAM dan een Pandas DataFrame
dtrain = xgb.DMatrix(X_train, label=y_train)
dval = xgb.DMatrix(X_val, label=y_val)

# VERWIJDER PANDAS DATA OM RAM LEEG TE MAKEN
# Dit voorkomt dat je data dubbel in het geheugen hebt staan
del X_train, X_val, y_train, y_val
gc.collect()

# ==========================================
# 4. PARAMETERS OPTIMALISEREN VOOR 8GB RAM
# ==========================================
# Bepaal het doel (objective)
objective = (
    "binary:logistic" if IS_CLASSIFICATION else "reg:squarederror"
)  # Gebruik 'multi:softprob' voor multiclass

params = {
    "objective": objective,
    # CRUCIAAL VOOR RAM: 'hist' bouwt histogrammen en gebruikt tot wel 10x minder geheugen
    "tree_method": "hist",
    # Beperk de diepte. Diepere bomen (bijv. >8) zorgen voor een explosie in RAM-gebruik
    "max_depth": 6,
    "learning_rate": 0.05,
    "eval_metric": "logloss" if IS_CLASSIFICATION else "rmse",
    "verbosity": 1,  # Toon waarschuwingen indien nodig
}

# ==========================================
# 5. MODEL TRAINEN
# ==========================================
print("4. Starten met trainen...")
evallist = [(dval, "validation"), (dtrain, "train")]
num_round = 1000  # Hoog aantal, we stoppen toch vroeg via early stopping

bst = xgb.train(
    params,
    dtrain,
    num_boost_round=num_round,
    evals=evallist,
    early_stopping_rounds=15,  # Stopt als model niet meer verbetert
    verbose_eval=50,  # Print status elke 50 bomen
)

print("\nTraining succesvol afgerond!")

# Optioneel: Model opslaan
# bst.save_model("xgboost_8gb_optimized.json")

1. Dtypes analyseren voor geheugenbesparing...
-> Data inladen met geoptimaliseerde dtypes...
2. Dataset opsplitsen...
3. DMatrix aanmaken (interne XGBoost structuur)...


ValueError: feature_names must be string, and may not contain [, ] or <